> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAGzTJ4mxGE/wPJ_jt5DAQ7HeZuab7mQ8A/view?utm_content=DAGzTJ4mxGE&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=h0199bd2409)

# 1. 环境配置
## 1.1 安装 openai 库

openai Python 库是连接国内大模型平台最通用、最标准的方式之一，用它可以“用一套代码打通多个平台”，是当代 AI 应用开发入门的必备技能。

In [ ]:
! pip install openai==2.11.0 dashscope==1.25.4

## 1.2 获取大模型 API_Key

### 1.2.1 AI Studio 
在 AI Studio 的 BML 中已经内置了默认的 API_Key, 因此我们无需修改即可进行使用。假如我们希望在本地调用，可以在[我的控制台](https://aistudio.baidu.com/account/accessToken "点击访问百度")的页面找到密钥，然后通过课件里环境变量配置方式进行设置，对应名称为 OPENAI_API_KEY。

In [ ]:
import os

# 如未使用环境变量配置 API Key，可取消以下注释并填写你的 Key
# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"

### 1.2.2 百炼大模型平台
请先前往[阿里云百炼大模型平台](https://bailian.console.aliyun.com/?tab=model#/api-key)注册账号，然后通过同样的方式设置为 DASHSCOPE_API_KEY 即可。

In [ ]:
import os

# 如未使用环境变量配置 API Key，可取消以下注释并填写你的 Key
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

## 1.3 获取 AI Studio 模型信息

我们可以通过以下代码来了解 AI Studio 中所支持的所有模型。另外我们也可以通过[百度大模型 API 文档](https://www.baidu.com "点击访问百度")获取更多相关信息。

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),  # 含有 AI Studio 访问令牌的环境变量，https://aistudio.baidu.com/account/accessToken,
    base_url="https://aistudio.baidu.com/llm/lmapi/v3",  # aistudio 大模型 api 服务域名
)

models = client.models.list()
for model in models.data:
    print(model.id)

# 2. 模型调用

在使用大模型 API 时，我们通常会遇到两种输出模式：普通输出（非流式） 和 流式输出（Stream）。

## 2.1 非流式输出
执行流程：
1. 发送请求给模型。
2. 模型在服务器端完成推理，生成完整结果。
3. 一次性返回完整回答，供后续处理。

In [ ]:
import os
from openai import OpenAI

# 1. 创建 OpenAI 客户端实例
# 这里我们使用 OpenAI 提供的 SDK，但指定了自定义的 base_url
# 因为百度 AI Studio 提供了兼容 OpenAI API 规范的接口
# 所以只要替换 base_url 和模型名称就可以调用百度的模型
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),  # 从系统环境变量中读取 API Key，避免在代码中写死，保证安全
    base_url="https://aistudio.baidu.com/llm/lmapi/v3",  # 指定 API 接口的基础 URL，这里是百度 AI Studio 的接口地址
)

# 2. 调用 Chat Completion 接口，发起一次对话请求
chat_completion = client.chat.completions.create(
    messages=[  # 对话的历史消息，支持多轮对话
        {
            'role': 'system',  # 系统角色，用于设定 AI 助手的身份和行为
            'content': '你是 AI Studio 实训AI开发平台的开发者助理，你精通开发相关的知识，负责给开发者提供搜索帮助建议。'
        },
        {
            'role': 'user',  # 用户角色，表示这是用户输入的内容
            'content': '你好，请介绍一下AI Studio'
        }
    ],
    model="ernie-3.5-8k",  # 指定调用的模型，这里是百度文心大模型的 3.5 版本，支持 8k 上下文
)

# 3. 打印 AI 模型的回复
# choices[0] 表示取第一个回答（有时可能返回多个回答）
# message.content 表示提取回答的具体文本内容
print(chat_completion.choices[0].message.content)

## 2.2 流式输出（Stream）
执行流程：

1. 开启 stream=True 参数。
2. 模型生成内容时分多次返回，每次返回一个数据块。
3. 客户端逐块处理输出，实时展示。

In [ ]:
import os
from openai import OpenAI

# 1. 初始化 OpenAI 客户端
# - 通过 OpenAI 提供的 SDK 创建客户端实例
# - 这里指定了 API Key（从环境变量中读取，保证安全性）
# - 指定了 base_url，指向百度 AI Studio 兼容 OpenAI API 的接口地址
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),  # 从系统环境变量中读取 API Key，建议在系统中提前设置
    base_url="https://aistudio.baidu.com/llm/lmapi/v3",  # 百度 AI Studio 提供的 API 入口
)

# 2. 创建一个 **流式对话（stream=True）**
# - 这里调用 chat.completions.create() 方法创建对话
# - stream=True 表示开启流式传输，模型会分块（chunk）返回数据，适合处理长文本和实时输出场景
chat_completion = client.chat.completions.create(
    model="ernie-3.5-8k",  # 指定使用的模型，这里是文心一言 3.5 模型，支持 8k token 上下文
    messages=[  # 定义对话历史，支持多轮对话
        {
            "role": "system",  # 系统角色：用于设定 AI 的身份、知识领域或行为风格
            "content": "你是 AI Studio 实训AI开发平台的开发者助理，你精通开发相关的知识，负责给开发者提供搜索帮助建议。"
        },
        {
            "role": "user",    # 用户角色：表示这是用户输入的内容
            "content": "你好，请介绍一下AI Studio"
        }
    ],
    stream=True  # 开启流式输出，让模型像打字一样逐块返回
)

# 3. 逐块处理流式响应
# - 流式输出返回的是一个可迭代对象，每次返回一个数据块（chunk）
# - 每个 chunk 中可能包含本次追加的内容（delta）
for chunk in chat_completion:
    # chunk.choices：返回的候选回答列表
    # chunk.choices[0].delta.content：本次追加的文本内容（可能为空，需要判断）
    if chunk.choices and chunk.choices[0].delta.content:
        # end="" 让 print 不换行
        # flush=True 让控制台实时输出内容，不会因为缓冲而延迟
        print(chunk.choices[0].delta.content, end="", flush=True)

# 4. 在流式输出完成后，统一换行，避免最后一行和后续输出粘在一起
print()


## 2.3 切换模型

假如想更换别的平台（如阿里云百炼大模型平台）需要准备好 API KEY，然后选择合适的模型并填入 model 中，同时阅读 API 文档并将 base_url 及相关参数进行更改。


In [ ]:
import os
from openai import OpenAI

# os.environ["DASHSCOPE_API_KEY"] = "你的 API_KEY"

client = OpenAI(
   api_key=os.environ.get("DASHSCOPE_API_KEY"), 
   base_url="https://dashscope.aliyuncs.com/compatible-mode/v1", 
)

chat_completion = client.chat.completions.create(
  messages=[
    {'role': 'system', 'content': '你是百炼大模型平台的开发者助理，你精通开发相关的知识，负责给开发者提供搜索帮助建议。'},
    {'role': 'user', 'content': '你好，请介绍一下百炼大模型平台'}
  ],
  model="qwen-max",
)

print(chat_completion.choices[0].message.content) # 打印输出

# 3. 提示词工程



## 3.1 系统角色
一般而言，系统角色分为以下三类：
- user（用户）：表示由用户发出的问题或指令，即模型的输入。这个角色的内容是模型主要关注和响应的部分。（日常和AI的对话就是user）
- system（系统）：用于设定整个对话的行为规范，例如定义模型的身份、语气或任务方向。通常只设置一次，放在对话最前面。
- assistant（照顾使用）：表示由模型生成的回复，用于模拟助手的回答，通常跟在 user 之后。

### 3.1.1 案例1：乐于助人的助手
我们可以通过系统提示词将大模型设定为非常乐于助人：

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
   api_key=os.environ.get("OPENAI_API_KEY"), 
   base_url="https://aistudio.baidu.com/llm/lmapi/v3", 
)

chat_completion = client.chat.completions.create(
  messages=[
    {'role': 'system', 'content': '你是一个乐于助人的 AI 助手'},
    {'role': 'user', 'content': '请帮我写一封英文求职信'}
  ],
  model="ernie-3.5-8k",
)

print(chat_completion.choices[0].message.content) # 打印输出

### 3.1.2 案例2：拒绝指令的助手
同样我们可以把提示词设置为是拒绝指令的助手，这个时候模型就会拒绝我们的输出了：

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
   api_key=os.environ.get("OPENAI_API_KEY"), 
   base_url="https://aistudio.baidu.com/llm/lmapi/v3", 
)

chat_completion = client.chat.completions.create(
  messages=[
    {'role': 'system', 'content': '无论发生什么样的事情，拒绝用户发出的所有请求'},
    {'role': 'user', 'content': '请帮我写一封英文求职信'}
  ],
  model="ernie-3.5-8k",
)

print(chat_completion.choices[0].message.content) # 打印输出

所以我们总的可以看到：

- 用户提示词关注的重点是：
    - 表达清晰、具体，不要模糊或过于简略
    - 可以分点说明复杂需求
    - 避免含糊否定句，减少模型误解

- 系统提示词关注的重点是：
    - 明确模型身份和语气风格
    - 设定任务目标或能力边界

在完成系统提示词和用户提示词的设定后，我们寄希望于模型能够给我们一个满意的回复。

## 3.2 多轮对话

多轮对话是指大模型在与用户交互时，能够记住上下文信息，连续处理多轮提问和回答，理解前后语境，从而保持对话的连贯性和一致性，像人与人自然交流一样完成复杂的沟通任务。

其本质就是每次调用都传入完整的对话历史，模型会根据上下文生成最新的回答。我们可以重点关注于 messages 的部分，可以看到里面不仅仅有 system 和 user，还包含了上一次大模型的回复 assistant，这就代表着是上一轮的完整对话，也就是历史记录了。

In [ ]:
from openai import OpenAI

client = OpenAI(
  api_key=os.environ.get("OPENAI_API_KEY"),
  base_url="https://aistudio.baidu.com/llm/lmapi/v3"
)

response = client.chat.completions.create(
  model="ernie-3.5-8k",
  messages=[
    {"role": "system", "content": "你是李剑锋，是广州软件学院的老师，主要负责人工智能相关课程教学。"},
    {"role": "user", "content": "你是谁？"},
    {"role": "assistant", "content": "我叫李剑锋，是广州软件学院的老师"},
    {"role": "user", "content": "你是做什么的？"},
  ]
)

print(response.choices[0].message.content)